In [0]:
%sql
CREATE OR REPLACE TABLE eindhoven.silver.meldingen AS 
SELECT 
    sha2(concat_ws('|', bron, onderwerp, subonderwerp, aangemaakt, opgelost, locatie), 256) AS melding_id, 
    bron, 
    onderwerp, 
    subonderwerp, 
    aangemaakt, 
    opgelost,
    CASE
        WHEN opgelost >= aangemaakt THEN datediff(day, aangemaakt, opgelost)
        END AS oplostijd_dagen,
    opgelost < aangemaakt AS is_tijd_ongeldig,
    buurtnaam, 
    buurtcode, 
    wijkcode, 
    wijknaam, 
    stadsdeelcode, 
    stadsdeelnaam, 
    locatie, 
    x_coord, 
    y_coord, 
    geo_point_2d
FROM eindhoven.bronze.meldingen
WHERE opgelost >= aangemaakt 
AND 1 = 0;

In [0]:
%sql

MERGE INTO eindhoven.silver.meldingen t USING (
    SELECT
        sha2(concat_ws('|', bron, onderwerp, subonderwerp, aangemaakt, opgelost, locatie), 256) AS melding_id, 
        bron, 
        onderwerp, 
        subonderwerp, 
        aangemaakt, 
        opgelost,
        CASE
            WHEN opgelost >= aangemaakt THEN datediff(day, aangemaakt, opgelost)
            END AS oplostijd_dagen,
        opgelost < aangemaakt AS is_tijd_ongeldig, 
        buurtnaam, 
        buurtcode, 
        wijkcode, 
        wijknaam, 
        stadsdeelcode, 
        stadsdeelnaam, 
        locatie, 
        x_coord, 
        y_coord, 
        geo_point_2d
FROM eindhoven.bronze.meldingen
) s ON t.melding_id = s.melding_id
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
SELECT
    COUNT(*),
    COUNT(*) FILTER (WHERE is_tijd_ongeldig = true),
    COUNT(*) FILTER (WHERE oplostijd_dagen IS NULL)

FROM eindhoven.silver.meldingen

In [0]:
%sql
SELECT count(*)
FROM eindhoven.silver.meldingen
WHERE